In [4]:
!git clone --recurse-submodules https://github.com/ServeurpersoCom/yue2.cpp
%cd yue2.cpp

# Baixar binários pré-compilados para a GPU T4 do Colab
!wget https://github.com/Harzur/YuE2-Q8_0-Colab-T4/releases/download/v1.0-t4/yue2_t4_binaries.zip -O /content/yue2_t4_binaries.zip

import os, zipfile
ruta_zip = "/content/yue2_t4_binaries.zip"
carpeta_en_zip = "build/"
destino = "/content/yue2.cpp/build"
os.makedirs(destino, exist_ok=True)

with zipfile.ZipFile(ruta_zip, "r") as z:
  for file_info in z.infolist():
    if file_info.filename.startswith(carpeta_en_zip) and not file_info.is_dir():
      archivo_relativo = file_info.filename[len(carpeta_en_zip):]
      if archivo_relativo:
        ruta_salida = os.path.join(destino, archivo_relativo)
        os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)
        with z.open(file_info) as fuente, open(ruta_salida, "wb") as salida:
          salida.write(fuente.read())

print("Binários configurados com sucesso!")

Cloning into 'yue2.cpp'...
remote: Enumerating objects: 1063, done.
remote: Counting objects: 100% (405/405), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 1063 (delta 342), reused 330 (delta 324), pack-reused 658 (from 1)
Receiving objects: 100% (1063/1063), 3.15 MiB | 8.13 MiB/s, done.
Resolving deltas: 100% (617/617), done.
Submodule 'ggml' (https://github.com/ServeurpersoCom/ggml.git) registered for path 'ggml'
Cloning into '/content/yue2.cpp/yue2.cpp/ggml'...
remote: Enumerating objects: 38156, done.        
remote: Counting objects: 100% (5099/5099), done.        
remote: Compressing objects: 100% (1074/1074), done.        
remote: Total 38156 (delta 4375), reused 4025 (delta 4025), pack-reused 33057 (from 4)        
Receiving objects: 100% (38156/38156), 34.14 MiB | 18.27 MiB/s, done.
Resolving deltas: 100% (27574/27574), done.
Submodule path 'ggml': checked out 'cc5df49a6b878db2b68b9a7b45976e279594627d'
/content/yue2.cpp/yue2.cpp
--2026-10-22 03:08:57--  

In [5]:
%cd /content/yue2.cpp
!pip install -q -U huggingface_hub --no-warn-conflicts
!chmod +x models.sh
!./models.sh

/content/yue2.cpp
[OK] YuE2-Vae-F32.gguf
[OK] YuE2-3B-Q8_0.gguf
[OK] MERT-v2-FullSong-Q8_0.gguf
[OK] SheetSage2-Q8_0.gguf
[OK] yue2-mothersuperior-realaudio-tokenizer-v4-Q8_0.gguf


In [ ]:
# Baixar o túnel para criar o link público
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

%cd /content/yue2.cpp
import subprocess, time, re, os

!chmod -R +x ./build/
!chmod +x ./cloudflared-linux-amd64

# Iniciar o túnel Cloudflare
log_file = open("tunnel.log", "w")
tunnel = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8087"],
                          stdout=log_file, stderr=log_file)

# Aguardar o link público aparecer
url = None
print("Gerando link seguro para a WebUI...")
for _ in range(15):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                url = match.group(0)
                break

if url:
    print(f"\n============================================================")
    print(f"👉 CLIQUE AQUI PARA ABRIR A WEBUI: {url}")
    print(f"============================================================\n")

# Iniciar o servidor
!./build/yue-server --host 0.0.0.0 --port 8087 \
    --model models/YuE2-3B-Q8_0.gguf \
    --vae models/YuE2-Vae-F32.gguf \
    --transcriber models/SheetSage2-Q8_0.gguf \
    --tokenizer models/yue2-mothersuperior-realaudio-tokenizer-v4-Q8_0.gguf > server.log 2>&1

/content/yue2.cpp
Gerando link seguro para a WebUI...

👉 CLIQUE AQUI PARA ABRIR A WEBUI: https://cow-drives-passage-jay.trycloudflare.com



In [ ]:
!cat /content/yue2.cpp/tunnel.log | grep -o 'https://.*\.trycloudflare\.com'